In [1]:
from bs4 import BeautifulSoup as bs
import dotenv
import os
from itemadapter import ItemAdapter
import psycopg2
from psycopg2.extras import RealDictCursor

In [2]:

dotenv.load_dotenv()

from pathlib import Path

# Carga robusta del archivo .env ubicado en la raíz del proyecto "scraper/.env",
# independientemente del directorio de ejecución.
THIS_DIR = Path(os.getcwd()).resolve()  # scraper/scraper
PROJECT_ROOT = THIS_DIR              # scraper/
ENV_PATH = PROJECT_ROOT / ".env"

# Si existe el .env en la raíz del proyecto, cárguelo explícitamente
dotenv.load_dotenv(dotenv_path=str(ENV_PATH))

# Usar variables de producción
DB_HOST = os.getenv('db_hostname_prod')
DB_USER = os.getenv('db_username_prod')
DB_PASSWORD = os.getenv('db_password_prod')
DB_NAME = os.getenv('db_name_prod')
DB_PORT = os.getenv('db_port_prod')
# Aceptar cualquiera de los dos nombres en .env por compatibilidad
DB_SSL_PATH = os.getenv('db_ssl_path') or os.getenv('db_path_ssl')

# Modo SSL configurable. Por defecto, usar 'require' (válido para AWS RDS).
# Si cuentas con el certificado de CA y quieres validación estricta,
# puedes establecer en .env: db_ssl_mode=verify-ca (o verify-full).
DB_SSL_MODE = os.getenv('db_ssl_mode', 'require')

# Validación mínima para ayudar a diagnosticar problemas de entorno
if not DB_HOST:
	# No imprimir secretos; solo pistas útiles.
	hint = f".env buscado en: {ENV_PATH} (exists={ENV_PATH.exists()})"
	raise RuntimeError(f"DB_HOST vacío: no se cargaron variables de entorno de producción. {hint}")

In [3]:
def _connect():
        """Establecer conexión a la base de datos"""
        hostname = DB_HOST
        username = DB_USER
        password = DB_PASSWORD
        database = DB_NAME
        port = DB_PORT

        # Preparar argumentos SSL según configuración
        ssl_args = {}
        if DB_SSL_MODE:
            ssl_args['sslmode'] = DB_SSL_MODE
        # Solo incluir sslrootcert si el modo requiere verificación
        if DB_SSL_MODE in ('verify-ca', 'verify-full') and DB_SSL_PATH:
            ssl_args['sslrootcert'] = DB_SSL_PATH

        connection = psycopg2.connect(
            host=hostname,
            user=username,
            password=password,
            dbname=database,
            port=port,
            **ssl_args
        )

        cur = connection.cursor(cursor_factory=RealDictCursor)
        cur.execute("SET search_path TO public;")

        return connection, cur
        

In [4]:
conn,cur = _connect()

OperationalError: connection to server at "scraper-db-instance.cgvssge2iuh4.us-east-1.rds.amazonaws.com" (3.81.255.15), port 5432 failed: Connection timed out (0x0000274C/10060)
	Is the server running on that host and accepting TCP/IP connections?


In [5]:
cur.execute("SELECT id, documento_url FROM legislacion_per where norma_completa is NULL;")
res = cur.fetchall()

In [6]:
print(res)

[RealDictRow({'id': 659, 'documento_url': 'https://spij.minjus.gob.pe/spij-ext-web/#/detallenorma/H1176874'}), RealDictRow({'id': 921, 'documento_url': 'https://spij.minjus.gob.pe/spij-ext-web/#/detallenorma/H910082'}), RealDictRow({'id': 1013, 'documento_url': 'https://spij.minjus.gob.pe/spij-ext-web/#/detallenorma/H1137181'}), RealDictRow({'id': 1054, 'documento_url': 'https://spij.minjus.gob.pe/spij-ext-web/#/detallenorma/H1212122'}), RealDictRow({'id': 1077, 'documento_url': 'https://spij.minjus.gob.pe/spij-ext-web/#/detallenorma/H1231121'}), RealDictRow({'id': 1157, 'documento_url': 'https://spij.minjus.gob.pe/spij-ext-web/#/detallenorma/H969850'})]


In [7]:
# Optional: Add explicit waits for specific elements to load
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By

In [8]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

service = Service(ChromeDriverManager().install())
  

In [9]:
XPATH_GET_ALL_LAWS_SALUD = "//div[@group='Salud']//font[@size='2']//a[1][text()!='']"
XPATH_GET_DATE_PUBLISHED = "//div[@class='ml-2 mt-4']/mat-label[1]/text()"
XPATH_GET_EMISOR_TEXT = "//*[@id='bodyContenido']//p[last()]/font[text()!='']/text()"
XPATH_GET_EMISOR = "//*[@id='bodyContenido']//p[last()]/font[text()!='']"
XPATH_GET_EMISOR2 = "//p[@style='font-style: normal; font-variant-ligatures: normal; font-variant-caps: normal; font-weight: normal; text-align: justify;']/font"
XPATH_GET_EMISOR2_TEXT = "//p[@style='font-style: normal; font-variant-ligatures: normal; font-variant-caps: normal; font-weight: normal; text-align: justify;']/font//text()"
XPATH_GET_COMPLETE_LAW = "//*[@id='bodyContenido']"
XPATH_GET_COMPLETE_LAW_TEXT = "//*[@id='bodyContenido']//text()"

SELECTOR_COMPLETE_LAW = "div#bodyContenido"


In [10]:
import time
from lxml import etree 
from selenium.common.exceptions import TimeoutException
from lxml.etree import _Element

In [11]:
def update_item(item):
    try:
        update_query = f"""
            UPDATE public.legislacion_per
            SET norma_completa = %s, emisor = %s, ano = %s
            WHERE id = %s;
        """
        cur.execute(
            update_query, (item['norma_completa'], item['emisor'], item['ano'], item['id']))
        conn.commit()
        print(
            f"Updated item ID {item['id']}  in the database.")
    except psycopg2.Error as e:
        print(
            f"Update error for item ID {item['id']}: {e}")
        conn.rollback()

In [12]:
def delete_item(id):
    try:
        update_query = f"""
            DELETE FROM public.legislacion_per
            WHERE id = %s;
        """
        cur.execute(
            update_query, (id, ))
        conn.commit()
        print(
            f"DELETE item ID {id}  in the database.")
    except psycopg2.Error as e:
        print(
            f"Update error for item ID {id}: {e}")
        conn.rollback()

In [13]:
def get_emisor(tree):

    match = tree.xpath(XPATH_GET_EMISOR2_TEXT)
    if len(match) == 0:
        match = tree.xpath(XPATH_GET_EMISOR_TEXT)
        if len(match) == 0:
            emisor = "N/A"
        else:
            if len(match[0]) > 500:
                emisor = "N/A"
            else:emisor = match[0]
    else: 
        emisor = match[0]
    
    return emisor
        

In [14]:
try:
    
    driver = webdriver.Chrome(service=service)  
    items = []
    for law in res:
        emisor = "0"
        driver.get(law["documento_url"])
        driver.refresh()
        try:
            WebDriverWait(driver, 7).until(EC.presence_of_element_located((By.XPATH, XPATH_GET_EMISOR)))
        except TimeoutException as _:
            driver.delete_all_cookies()
        soup = bs(driver.page_source, 'html5lib')
        tree = etree.HTML(str(soup))
        norma = tree.xpath(XPATH_GET_COMPLETE_LAW_TEXT)
        norma_completa = ' '.join([text.strip()
                                    for text in norma]).strip()
        if "derogad" in norma_completa.lower() :
            print("Norma derogada, continuando")
            delete_item(law["id"])
            continue
        emisor = get_emisor(tree)
        date = tree.xpath(XPATH_GET_DATE_PUBLISHED)[-1]
        ano = date.split(" ")[-1]
        item = {"id":law["id"], "norma_completa": norma_completa, "emisor":emisor.strip(), "ano":ano.strip()}
        
        update_item(item)
        time.sleep(0.5)
finally:
    driver.close()
    

Updated item ID 659  in the database.
Updated item ID 921  in the database.
Updated item ID 1013  in the database.
Updated item ID 1054  in the database.
Update error for item ID 1077: invalid input syntax for type integer: ""
LINE 3: ...           SET norma_completa = '', emisor = 'N/A', ano = ''
                                                                     ^

Updated item ID 1157  in the database.
